
# 🩺 Diabetes Progression Prediction
## Data Science Internship Project

### Dataset: `load_diabetes()` from Scikit-learn

This project predicts the quantitative disease progression score using patient health measurements.

The analysis begins with the required exploratory work and then builds regression models from simple to more advanced:

1. Simple Linear Regression
2. Multiple Linear Regression
3. Decision Tree Regression
4. Random Forest Regression
5. KNN Regression

Model performance is evaluated using **R², MAE and RMSE**.



# 1. Project Introduction

Diabetes progression prediction is treated as a regression problem in which the objective is to estimate a continuous disease progression score from patient health measurements.

The project uses the built-in `load_diabetes()` dataset available in Scikit-learn.



# 2. Problem Statement

The objective is to analyze patient health measurements and develop regression models capable of predicting the disease progression score after one year.

The project first establishes a simple linear baseline, then evaluates whether using multiple features and nonlinear/machine-learning regression methods can improve prediction performance.



# 3. Objectives

- Load and understand the Scikit-learn diabetes dataset.
- Perform exploratory data analysis.
- Study feature correlations.
- Identify influential features.
- Build a Simple Linear Regression model.
- Build a Multiple Linear Regression model.
- Build Decision Tree, Random Forest and KNN regression models.
- Evaluate all models using R², MAE and RMSE.
- Compare model performance.
- Select the best-performing model.
- Analyze actual vs predicted values and residual errors.


# 4. Import Libraries

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("Libraries imported successfully.")


# 5. Load Dataset

In [ ]:

# Load the built-in diabetes dataset
diabetes = load_diabetes(as_frame=True)

# Convert to Pandas DataFrame
df = diabetes.frame.copy()

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

display(df.head())


# 6. Dataset Overview

In [ ]:

print("Number of rows   :", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))



# 7. Feature Description

The dataset contains 10 input features and 1 target variable.

| Feature | Description |
|---|---|
| `age` | Age-related standardized measurement |
| `sex` | Sex-related standardized measurement |
| `bmi` | Body Mass Index-related standardized measurement |
| `bp` | Blood pressure-related standardized measurement |
| `s1` | First blood serum measurement |
| `s2` | Second blood serum measurement |
| `s3` | Third blood serum measurement |
| `s4` | Fourth blood serum measurement |
| `s5` | Fifth blood serum measurement |
| `s6` | Sixth blood serum measurement |
| `target` | Quantitative disease progression score after one year |

**Note:** The Scikit-learn dataset contains standardized feature values.


# 8. Data Quality Check

In [ ]:

print("Missing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("Total missing values:", df.isnull().sum().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique values:")
display(df.nunique().to_frame("Unique Values"))


# 9. Descriptive Statistics

In [ ]:

display(df.describe().T)


# 10. Exploratory Data Analysis

## 10.1 Target Distribution

In [ ]:

plt.figure(figsize=(9, 5))
plt.hist(df["target"], bins=25, edgecolor="black")
plt.title("Distribution of Diabetes Progression Score")
plt.xlabel("Disease Progression Score")
plt.ylabel("Frequency")
plt.grid(alpha=0.2)
plt.show()


## 10.2 Feature Distributions

In [ ]:

features = [column for column in df.columns if column != "target"]

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
axes = axes.ravel()

for i, feature in enumerate(features):
    axes[i].hist(df[feature], bins=20, edgecolor="black")
    axes[i].set_title(f"Distribution of {feature}")
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel("Frequency")

for i in range(len(features), len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## 10.3 Boxplots

In [ ]:

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.ravel()

for i, feature in enumerate(features):
    axes[i].boxplot(df[feature])
    axes[i].set_title(f"Boxplot of {feature}")
    axes[i].set_ylabel("Value")

for i in range(len(features), len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## 10.4 Feature vs Target

In [ ]:

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
axes = axes.ravel()

for i, feature in enumerate(features):
    axes[i].scatter(df[feature], df["target"], alpha=0.65)
    axes[i].set_title(f"{feature} vs Target")
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel("Target")

for i in range(len(features), len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()


# 11. Correlation Analysis

In [ ]:

correlation_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
plt.imshow(correlation_matrix, cmap="coolwarm", aspect="auto")
plt.colorbar(label="Correlation")

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45,
    ha="right"
)
plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns
)

for i in range(len(correlation_matrix.columns)):
    for j in range(len(correlation_matrix.columns)):
        plt.text(
            j, i,
            f"{correlation_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

target_correlations = (
    correlation_matrix["target"]
    .drop("target")
    .sort_values(key=abs, ascending=False)
)

display(target_correlations.to_frame("Correlation with Target"))


# 12. Important Feature Analysis

In [ ]:

important_features = pd.DataFrame({
    "Feature": target_correlations.index,
    "Correlation": target_correlations.values,
    "Absolute Correlation": target_correlations.abs().values
}).sort_values("Absolute Correlation", ascending=False).reset_index(drop=True)

display(important_features)

plt.figure(figsize=(9, 6))
plt.barh(
    important_features["Feature"],
    important_features["Absolute Correlation"]
)
plt.xlabel("Absolute Correlation")
plt.ylabel("Feature")
plt.title("Feature Relationship with Target")
plt.gca().invert_yaxis()
plt.show()

# Select the strongest feature for Simple Linear Regression.
simple_feature = important_features.iloc[0]["Feature"]
print("Feature selected for Simple Linear Regression:", simple_feature)


# 13. Data Preparation

In [ ]:

X = df.drop(columns=["target"])
y = df["target"]

print("Input features:", X.columns.tolist())
print("Target:", "target")


# 14. Train-Test Split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])



# 15. Simple Linear Regression

Simple Linear Regression uses **one independent feature** to predict the target.

The feature selected here is the feature with the strongest absolute correlation with the target, identified in the Important Feature Analysis section.


## 15.1 Training

In [ ]:

simple_model = LinearRegression()

# Use only the selected single feature
simple_model.fit(
    X_train[[simple_feature]],
    y_train
)

print("Simple Linear Regression trained using:", simple_feature)


## 15.2 Prediction

In [ ]:

simple_predictions = simple_model.predict(
    X_test[[simple_feature]]
)

simple_prediction_table = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": simple_predictions
})

display(simple_prediction_table.head(10))


## 15.3 R², MAE, RMSE

In [ ]:

simple_r2 = r2_score(y_test, simple_predictions)
simple_mae = mean_absolute_error(y_test, simple_predictions)
simple_rmse = np.sqrt(mean_squared_error(y_test, simple_predictions))

print(f"R²   : {simple_r2:.4f}")
print(f"MAE  : {simple_mae:.4f}")
print(f"RMSE : {simple_rmse:.4f}")


## 15.4 Coefficient Analysis

In [ ]:

print("Intercept:", simple_model.intercept_)
print("Coefficient:", simple_model.coef_[0])

plt.figure(figsize=(8, 5))
plt.scatter(
    X_test[simple_feature],
    y_test,
    alpha=0.7,
    label="Actual"
)

sorted_x = np.sort(X_test[simple_feature].values)
sorted_pred = simple_model.predict(
    sorted_x.reshape(-1, 1)
)

plt.plot(
    sorted_x,
    sorted_pred,
    linestyle="--",
    label="Regression Line"
)

plt.xlabel(simple_feature)
plt.ylabel("Target")
plt.title(f"Simple Linear Regression: {simple_feature} vs Target")
plt.legend()
plt.grid(alpha=0.2)
plt.show()



# 16. Multiple Linear Regression

Multiple Linear Regression uses **all available input features together** to predict the target.

This provides a stronger baseline than Simple Linear Regression because the model can use information from all 10 features.


## 16.1 Training

In [ ]:

multiple_model = LinearRegression()
multiple_model.fit(X_train, y_train)

print("Multiple Linear Regression trained successfully.")


## 16.2 Prediction

In [ ]:

multiple_predictions = multiple_model.predict(X_test)

display(pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": multiple_predictions
}).head(10))


## 16.3 R², MAE, RMSE

In [ ]:

multiple_r2 = r2_score(y_test, multiple_predictions)
multiple_mae = mean_absolute_error(y_test, multiple_predictions)
multiple_rmse = np.sqrt(mean_squared_error(y_test, multiple_predictions))

print(f"R²   : {multiple_r2:.4f}")
print(f"MAE  : {multiple_mae:.4f}")
print(f"RMSE : {multiple_rmse:.4f}")


## 16.4 Coefficient Analysis

In [ ]:

multiple_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": multiple_model.coef_
})

multiple_coefficients["Absolute Coefficient"] = (
    multiple_coefficients["Coefficient"].abs()
)

multiple_coefficients = multiple_coefficients.sort_values(
    "Absolute Coefficient",
    ascending=False
)

display(multiple_coefficients)

plt.figure(figsize=(9, 6))
plt.barh(
    multiple_coefficients["Feature"],
    multiple_coefficients["Coefficient"]
)
plt.axvline(0, linewidth=1)
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.title("Multiple Linear Regression Coefficients")
plt.gca().invert_yaxis()
plt.show()



# 17. Decision Tree Regression

Decision Tree Regression learns nonlinear relationships by dividing the feature space into decision-based regions.


## 17.1 Training

In [ ]:

decision_tree_model = DecisionTreeRegressor(
    random_state=42,
    max_depth=4
)

decision_tree_model.fit(X_train, y_train)

print("Decision Tree Regression trained successfully.")


## 17.2 Prediction

In [ ]:

decision_tree_predictions = decision_tree_model.predict(X_test)

display(pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": decision_tree_predictions
}).head(10))


## 17.3 Evaluation

In [ ]:

decision_tree_r2 = r2_score(y_test, decision_tree_predictions)
decision_tree_mae = mean_absolute_error(y_test, decision_tree_predictions)
decision_tree_rmse = np.sqrt(mean_squared_error(y_test, decision_tree_predictions))

print(f"R²   : {decision_tree_r2:.4f}")
print(f"MAE  : {decision_tree_mae:.4f}")
print(f"RMSE : {decision_tree_rmse:.4f}")



# 18. Random Forest Regression

Random Forest Regression combines multiple decision trees to produce a more stable prediction than a single decision tree.


## 18.1 Training

In [ ]:

random_forest_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

random_forest_model.fit(X_train, y_train)

print("Random Forest Regression trained successfully.")


## 18.2 Prediction

In [ ]:

random_forest_predictions = random_forest_model.predict(X_test)

display(pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": random_forest_predictions
}).head(10))


## 18.3 Evaluation

In [ ]:

random_forest_r2 = r2_score(y_test, random_forest_predictions)
random_forest_mae = mean_absolute_error(y_test, random_forest_predictions)
random_forest_rmse = np.sqrt(mean_squared_error(y_test, random_forest_predictions))

print(f"R²   : {random_forest_r2:.4f}")
print(f"MAE  : {random_forest_mae:.4f}")
print(f"RMSE : {random_forest_rmse:.4f}")



# 19. KNN Regression

K-Nearest Neighbors Regression predicts a new observation using the target values of nearby training observations.

Because KNN is distance-based, the features are standardized before training.


## 19.1 Training

In [ ]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_model = KNeighborsRegressor(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

print("KNN Regression trained successfully.")


## 19.2 Prediction

In [ ]:

knn_predictions = knn_model.predict(X_test_scaled)

display(pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": knn_predictions
}).head(10))


## 19.3 Evaluation

In [ ]:

knn_r2 = r2_score(y_test, knn_predictions)
knn_mae = mean_absolute_error(y_test, knn_predictions)
knn_rmse = np.sqrt(mean_squared_error(y_test, knn_predictions))

print(f"R²   : {knn_r2:.4f}")
print(f"MAE  : {knn_mae:.4f}")
print(f"RMSE : {knn_rmse:.4f}")



# 20. Model Performance Comparison

All five regression models are compared using the same test set.

### Models
- Simple Linear Regression
- Multiple Linear Regression
- Decision Tree Regression
- Random Forest Regression
- KNN Regression

### Metrics
- **R²:** Higher is better
- **MAE:** Lower is better
- **RMSE:** Lower is better


In [ ]:

results_df = pd.DataFrame([
    {
        "Model": "Simple Linear Regression",
        "R²": simple_r2,
        "MAE": simple_mae,
        "RMSE": simple_rmse
    },
    {
        "Model": "Multiple Linear Regression",
        "R²": multiple_r2,
        "MAE": multiple_mae,
        "RMSE": multiple_rmse
    },
    {
        "Model": "Decision Tree Regression",
        "R²": decision_tree_r2,
        "MAE": decision_tree_mae,
        "RMSE": decision_tree_rmse
    },
    {
        "Model": "Random Forest Regression",
        "R²": random_forest_r2,
        "MAE": random_forest_mae,
        "RMSE": random_forest_rmse
    },
    {
        "Model": "KNN Regression",
        "R²": knn_r2,
        "MAE": knn_mae,
        "RMSE": knn_rmse
    }
])

display(results_df.round(4))


### 20.1 R² Comparison

In [ ]:

plt.figure(figsize=(11, 5))
plt.bar(results_df["Model"], results_df["R²"])
plt.title("R² Comparison — Higher is Better")
plt.ylabel("R²")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


### 20.2 MAE Comparison

In [ ]:

plt.figure(figsize=(11, 5))
plt.bar(results_df["Model"], results_df["MAE"])
plt.title("MAE Comparison — Lower is Better")
plt.ylabel("MAE")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


### 20.3 RMSE Comparison

In [ ]:

plt.figure(figsize=(11, 5))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.title("RMSE Comparison — Lower is Better")
plt.ylabel("RMSE")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()



# 21. Best Model Selection

For regression, there is no single "accuracy" percentage.

The best model should generally have:
- Higher **R²**
- Lower **MAE**
- Lower **RMSE**

A combined ranking is used below to make the selection transparent.


In [ ]:

ranking = results_df.copy()

ranking["R² Rank"] = ranking["R²"].rank(
    ascending=False,
    method="min"
)

ranking["MAE Rank"] = ranking["MAE"].rank(
    ascending=True,
    method="min"
)

ranking["RMSE Rank"] = ranking["RMSE"].rank(
    ascending=True,
    method="min"
)

ranking["Combined Rank"] = (
    ranking["R² Rank"] +
    ranking["MAE Rank"] +
    ranking["RMSE Rank"]
)

ranking = ranking.sort_values(
    ["Combined Rank", "R²"],
    ascending=[True, False]
).reset_index(drop=True)

display(ranking.round(4))

best_model_name = ranking.iloc[0]["Model"]
print("Best overall model:", best_model_name)



# 22. Actual vs Predicted

The actual-vs-predicted plot shows how closely the model predictions follow the real target values.

Points closer to the diagonal reference line indicate better prediction agreement.


In [ ]:

prediction_map = {
    "Simple Linear Regression": simple_predictions,
    "Multiple Linear Regression": multiple_predictions,
    "Decision Tree Regression": decision_tree_predictions,
    "Random Forest Regression": random_forest_predictions,
    "KNN Regression": knn_predictions
}

best_predictions = prediction_map[best_model_name]

plt.figure(figsize=(8, 6))
plt.scatter(
    y_test,
    best_predictions,
    alpha=0.7
)

minimum = min(y_test.min(), best_predictions.min())
maximum = max(y_test.max(), best_predictions.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual Disease Progression")
plt.ylabel("Predicted Disease Progression")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.grid(alpha=0.2)
plt.show()



# 23. Residual / Error Analysis

Residual = **Actual Value − Predicted Value**

A good regression model should generally have residuals distributed around zero without a strong systematic pattern.


In [ ]:

residuals = y_test.values - best_predictions

plt.figure(figsize=(9, 5))
plt.scatter(
    best_predictions,
    residuals,
    alpha=0.7
)
plt.axhline(0, linestyle="--")

plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title(f"Residual Analysis — {best_model_name}")
plt.grid(alpha=0.2)
plt.show()


In [ ]:

error_analysis = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": best_predictions,
    "Residual": residuals,
    "Absolute Error": np.abs(residuals)
}).sort_values(
    "Absolute Error",
    ascending=False
)

print("Largest prediction errors:")
display(error_analysis.head(10))



# 24. Feature Importance

For tree-based models, feature importance is obtained directly from the trained model.

For Multiple Linear Regression, the absolute coefficient magnitude provides an interpretable measure of feature influence.

The section below reports feature importance for the best model.


In [ ]:

if best_model_name == "Decision Tree Regression":
    importance_values = decision_tree_model.feature_importances_
    importance_method = "Decision Tree feature importance"

elif best_model_name == "Random Forest Regression":
    importance_values = random_forest_model.feature_importances_
    importance_method = "Random Forest feature importance"

elif best_model_name == "Multiple Linear Regression":
    importance_values = np.abs(multiple_model.coef_)
    importance_method = "Absolute Linear Regression coefficients"

elif best_model_name == "Simple Linear Regression":
    importance_values = np.zeros(len(X.columns))
    importance_values[list(X.columns).index(simple_feature)] = abs(
        simple_model.coef_[0]
    )
    importance_method = "Simple Linear Regression coefficient"

else:
    # KNN does not provide native feature_importances_.
    # Use the earlier correlation-based analysis instead.
    importance_values = important_features.set_index("Feature").reindex(X.columns)["Absolute Correlation"].values
    importance_method = "Absolute feature-target correlation (KNN has no native feature importance)"

feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance_values
}).sort_values(
    "Importance",
    ascending=False
)

print("Importance method:", importance_method)
display(feature_importance_df)

plt.figure(figsize=(9, 6))
plt.barh(
    feature_importance_df["Feature"],
    feature_importance_df["Importance"]
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(f"Feature Importance — {best_model_name}")
plt.gca().invert_yaxis()
plt.show()



# 25. Key Findings

After running the notebook, summarize the actual results using the generated tables and graphs.

### Points to report

- The dataset contains 442 observations and 10 input features.
- Data quality was checked for missing values and duplicates.
- Exploratory analysis was performed for the target and all input features.
- Feature correlations were analyzed to identify relationships with the target.
- Simple Linear Regression provided a one-feature baseline.
- Multiple Linear Regression used all available features.
- Decision Tree, Random Forest and KNN provided additional regression approaches.
- R², MAE and RMSE were used for model evaluation.
- The best-performing model was selected using the combined metric ranking.
- Actual-vs-predicted and residual analysis were used to inspect prediction behavior.

**Important:** Feature correlation or model importance does not prove medical causation.



# 26. Limitations

- The dataset is relatively small for a real-world healthcare application.
- The Scikit-learn dataset contains standardized feature values.
- Results depend on the selected train-test split.
- Model performance on this dataset does not guarantee performance on external data.
- Correlation and feature importance represent predictive relationships, not causation.
- This project should not be used as a clinical diagnostic system.



# 27. Future Scope

Possible future improvements include:

- Hyperparameter tuning for Decision Tree, Random Forest and KNN.
- K-fold cross-validation.
- Testing the models on an independent dataset.
- More detailed error analysis.
- Explainable AI methods such as SHAP.
- Deployment as a simple web application.
- Monitoring model performance on new data.



# Final Conclusion

This project demonstrates an end-to-end regression workflow using the **Scikit-learn `load_diabetes()` dataset**.

The analysis progresses from a **Simple Linear Regression baseline** to **Multiple Linear Regression**, followed by **Decision Tree, Random Forest and KNN Regression**.

The models are evaluated using **R², MAE and RMSE**, compared on the same test set, and further analyzed using actual-vs-predicted plots, residual analysis and feature importance.

The final results should be interpreted from the values generated by running the notebook rather than from predetermined accuracy claims.
